# Phase 2 — The Debate System

Four LLMs collaborate: three **Solvers** answer independently, peer-review,
refine, and a fourth **Judge** picks the best answer.

**Pipeline** (`debate.pipeline`): Role selection → Round 1 → Peer review →
Round 2 → Judgment. The batch runner lives in `debate.runner`.

The walkthrough below reads a **saved run** (no key). The clearly-marked live
cells run real debates when `RUN_LIVE = True`.

In [1]:
# --- setup: make the in-repo packages importable; resolve project paths ---
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))       # the `debate` package
sys.path.insert(0, str(ROOT / "scripts"))   # build_dashboard (dashboard generator)

DATA = ROOT / "data"
OUTPUTS = ROOT / "outputs"

In [2]:
# Async + live-call setup. Notebooks share the kernel's event loop, so
# `nest_asyncio` lets the package's asyncio.run(...) calls work here.
import os
import asyncio
import nest_asyncio
nest_asyncio.apply()

RUN_LIVE = False   # set True (needs OPEN_ROUTER_KEY in .env) to run real API calls
if RUN_LIVE:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
HAS_KEY = bool(os.getenv("OPEN_ROUTER_KEY"))
print(f"RUN_LIVE={RUN_LIVE}  API key available={HAS_KEY}")

RUN_LIVE=False  API key available=False


In [3]:
import json
from debate.models import load_config, create_model_clients

config = load_config()
print("Participant pool (config/models.toml):")
for slot, spec in config.participants.items():
    print(f"  {slot}: {spec.model}")
print("Grader:", config.grader.model)

Participant pool (config/models.toml):
  model_a: google/gemini-3-flash-preview
  model_b: openai/gpt-4o-mini
  model_c: google/gemma-3-27b-it
  model_d: meta-llama/llama-3.1-8b-instruct
Grader: google/gemini-2.5-flash


In [4]:
def parse(v):   # saved runs may store nested fields as JSON strings
    return json.loads(v) if isinstance(v, str) else v

def to_result(r):
    from debate.schemas import DebateResult
    r = dict(r)
    for k in ("role_preferences", "solutions", "refined_solutions"):
        r[k] = {kk: parse(vv) for kk, vv in r[k].items()}
    r["judgment"] = parse(r["judgment"])
    return DebateResult(**r)

## One problem through the pipeline (from a saved run)

In [5]:
run = json.loads((OUTPUTS / "hard_result.json").read_text())
res = next((r for r in run["results"] if r["problem_id"] == "hard_024"), run["results"][0])
pid = res["problem_id"]
problem = next(p for p in json.loads((DATA / "llm_hard_problems.json").read_text()) if p["id"] == pid)
print(pid, "-", problem["question"])
print("correct answer:", problem["correct_answer"])

hard_024 - How many pairs of twins do you need in a room for there to be at least a 50% chance that two people have the same birthday?
correct answer: 1


### Stage 0 — Role selection & the deterministic assignment algorithm

In [6]:
from debate.pipeline import assign_roles
from debate.schemas import RolePreferenceResponse

prefs = {slot: RolePreferenceResponse(**parse(p)) for slot, p in res["role_preferences"].items()}
for slot, p in prefs.items():
    print(f"{slot}: wants {p.preferred_role:6s} (solver {p.solver_confidence}, judge {p.judge_confidence})")

# assign_roles is deterministic: highest judge-confidence becomes the Judge.
roles = assign_roles({s: None for s in prefs}, prefs)
print("\nassign_roles ->", roles)
print("matches the saved run:", roles == res["roles"])

model_a: wants Solver (solver 0.8, judge 0.6)
model_b: wants Solver (solver 0.82, judge 0.68)
model_c: wants Solver (solver 0.85, judge 0.65)
model_d: wants Solver (solver 0.85, judge 0.6)

assign_roles -> {'solver_1': 'model_a', 'solver_2': 'model_c', 'solver_3': 'model_d', 'judge': 'model_b'}
matches the saved run: True


### Stage 1 — Independent solutions

In [7]:
for sid, sol in res["solutions"].items():
    print(f"{sid}: {parse(sol)['answer']!r}")

solver_1: '23'
solver_2: '23'
solver_3: '1'


### Stage 2 — Peer review (each solution reviewed by the other two)

In [8]:
for sid, reviews in res["peer_reviews"].items():
    print(f"reviews of {sid}: {len(reviews)}")
print("\nexample review:\n", res["peer_reviews"][list(res["peer_reviews"])[0]][0][:300])

reviews of solver_1: 2
reviews of solver_2: 2
reviews of solver_3: 2

example review:
 Strengths:
* **Correctly identifies the core concept:** The solution correctly recognizes that this is a variation of the birthday problem.
* **Clear explanation of the probability calculation:** The formula for `P(no shared birthday)` is well-explained and accurate.
* **Explicitly states the goal:*


### Stage 3 — Refinement

In [9]:
for sid, ref in res["refined_solutions"].items():
    print(f"{sid}: {parse(ref)['refined_answer']!r}")

solver_1: '23'
solver_2: '1'
solver_3: '1'


### Stage 4 — Judgment

In [10]:
j = parse(res["judgment"])
print("winner:", j["winner"], "| confidence:", j["confidence"])
print("reasoning:", j["reasoning"][:280])

winner: solver_3 | confidence: 0.92
reasoning: Solver 3 correctly identifies the trick nature of the problem and provides the accurate answer of 1 pair of twins. By definition, twins share the same birthday, so a single pair immediately satisfies the condition that 'two people have the same birthday' with 100% certainty, exce


## Run it live

`run_debate_for_problem` runs one problem through the real pipeline;
`run_all_debates` runs a whole batch **concurrently**. Set `RUN_LIVE = True`
above (needs `OPEN_ROUTER_KEY`).

In [11]:
from debate.problem import Problem
from debate.pipeline import run_debate_for_problem

if RUN_LIVE and HAS_KEY:
    models = create_model_clients(config)
    live = asyncio.run(run_debate_for_problem(Problem(**problem), models))
    print("roles:", live.roles)
    print("judge picked:", live.judgment.winner, "->",
          live.refined_solutions[live.judgment.winner].refined_answer)
else:
    print("RUN_LIVE is off — skipping the live single-problem debate.")

RUN_LIVE is off — skipping the live single-problem debate.


In [12]:
from debate.runner import load_problems, run_all_debates

if RUN_LIVE and HAS_KEY:
    models = create_model_clients(config)
    batch = load_problems("llm_hard_problems.json")[:2]      # 2 problems, run concurrently
    results = asyncio.run(run_all_debates(batch, models, max_concurrent=2))
    for r in results:
        print(r.problem_id, "-> judge picked", r.judgment.winner)
else:
    print("RUN_LIVE is off — skipping the concurrent batch run.")

RUN_LIVE is off — skipping the concurrent batch run.


## The LLM grader (free-answer equivalence)

In [13]:
from debate.models import make_llm_grader, create_grader_client

if RUN_LIVE and HAS_KEY:
    grade = make_llm_grader(create_grader_client(config))
    print("grade(Q, 'a single race', 'Race all six together'):",
          grade("Best way to race six horses?", "Just do one race with all six", "Race all six together in a single race."))
else:
    print("RUN_LIVE is off — the grader makes an API call, so it is skipped here.")

RUN_LIVE is off — the grader makes an API call, so it is skipped here.
